# Lesson 7 — Acquisition Functions: PI, EI, UCB

We have a GP surrogate with posterior mean $\mu(x)$ and standard deviation $\sigma(x)$. The next question is: **where should we evaluate the expensive function next?**

An **acquisition function** $\alpha(x)$ scores every candidate point using $\mu(x)$ and $\sigma(x)$. We evaluate $f$ at:

$$x_{\text{next}} = \arg\max_x \; \alpha(x)$$

The three most common acquisition functions:

| Name | Formula (minimization) | Trade-off knob |
|---|---|---|
| **PI** | $\Phi\!\left(\frac{f^* - \xi - \mu}{\sigma}\right)$ | $\xi$ |
| **EI** | $(f^* - \xi - \mu)\Phi(Z) + \sigma\phi(Z)$ | $\xi$ |
| **LCB** | $-(\mu - \kappa\sigma)$ | $\kappa$ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

## Benchmark, kernel, and GP (from Lesson 6)

In [ ]:
def forrester(x):
    x = np.asarray(x).ravel()
    return (6*x - 2)**2 * np.sin(12*x - 4)

def matern52(X1, X2, length_scale=1.0, signal_var=1.0):
    X1 = np.atleast_2d(X1); X2 = np.atleast_2d(X2)
    diff = X1[:, None, :] - X2[None, :, :]
    r = np.sqrt(np.sum(diff**2, axis=-1))
    s = np.sqrt(5) * r / length_scale
    return signal_var * (1 + s + s**2/3) * np.exp(-s)

def gp_predict(X_train, y_train, X_test, kernel_fn, noise=1e-4):
    K    = kernel_fn(X_train, X_train) + noise * np.eye(len(X_train))
    Ks   = kernel_fn(X_test, X_train)
    Kss  = kernel_fn(X_test, X_test)
    L    = np.linalg.cholesky(K)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_train))
    mu   = Ks @ alpha
    v    = np.linalg.solve(L, Ks.T)
    std  = np.sqrt(np.maximum(np.diag(Kss) - np.sum(v**2, axis=0), 0))
    return mu, std

kern = lambda X1, X2: matern52(X1, X2, length_scale=0.25, signal_var=4.0)

x_fine  = np.linspace(0, 1, 300).reshape(-1, 1)
xs      = x_fine.squeeze()
y_true  = forrester(xs)

X_train = np.array([[0.1], [0.3], [0.5], [0.7], [0.9]])
y_train = forrester(X_train)
f_best  = y_train.min()

mu, std = gp_predict(X_train, y_train, x_fine, kern, noise=0.01)
print(f"f_best (current best observed): {f_best:.4f}")

## Acquisition functions — implementations

In [ ]:
def acq_pi(mu, std, f_best, xi=0.0):
    """Probability of Improvement (minimization)."""
    Z = (f_best - xi - mu) / (std + 1e-9)
    return norm.cdf(Z)

def acq_ei(mu, std, f_best, xi=0.01):
    """Expected Improvement (minimization)."""
    Z = (f_best - xi - mu) / (std + 1e-9)
    return np.maximum((f_best - xi - mu) * norm.cdf(Z) + std * norm.pdf(Z), 0)

def acq_lcb(mu, std, kappa=2.0):
    """Negative Lower Confidence Bound — argmax gives the LCB minimizer."""
    return -(mu - kappa * std)

# Quick sanity check
pi_test  = acq_pi(mu, std, f_best)
ei_test  = acq_ei(mu, std, f_best)
lcb_test = acq_lcb(mu, std)
print(f"PI range:  [{pi_test.min():.3f}, {pi_test.max():.3f}]")
print(f"EI range:  [{ei_test.min():.3f}, {ei_test.max():.3f}]")
print(f"−LCB range: [{lcb_test.min():.3f}, {lcb_test.max():.3f}]")

## Helper: 2-row GP + acquisition plot

In [ ]:
def draw_panel(ax_gp, ax_acq, mu, std, acq_vals, title, acq_label):
    x_next = x_fine[np.argmax(acq_vals)].item()

    ax_gp.plot(xs, y_true, color="gray", linestyle="--", linewidth=1.5, alpha=0.4, label="True")
    ax_gp.plot(xs, mu, color="steelblue", linewidth=2, label="GP μ")
    ax_gp.fill_between(xs, mu-2*std, mu+2*std, alpha=0.2, color="steelblue", label="±2σ")
    ax_gp.scatter(X_train.squeeze(), y_train, color="black", s=50, zorder=5)
    ax_gp.axhline(f_best, color="gray", linewidth=1, linestyle=":", alpha=0.6)
    ax_gp.axvline(x_next, color="red", linewidth=2, linestyle=":",
                  label=f"x_next={x_next:.3f}")
    ax_gp.set_title(title)
    ax_gp.set_ylabel("f(x)")
    ax_gp.legend(fontsize=6)
    ax_gp.grid(True, alpha=0.3)

    ax_acq.plot(xs, acq_vals, color="darkorange", linewidth=2)
    ax_acq.fill_between(xs, 0, acq_vals, alpha=0.25, color="darkorange")
    ax_acq.axvline(x_next, color="red", linewidth=2, linestyle=":")
    ax_acq.set_ylabel(acq_label)
    ax_acq.set_xlabel("x")
    ax_acq.grid(True, alpha=0.3)

print("draw_panel ready.")

## Part 1 — Why acquisition functions?

The GP gives us two signals at every point:
- **Low μ(x)** → the surrogate thinks f(x) is small → good for exploitation
- **High σ(x)** → the surrogate is uncertain → good for exploration

Evaluating only at low μ risks getting stuck. Evaluating only at high σ ignores what we already know. Acquisition functions combine both.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(xs, y_true, color="gray", linestyle="--", linewidth=1.5, alpha=0.5, label="True Forrester")
ax.plot(xs, mu, color="steelblue", linewidth=2, label="GP mean μ(x)")
ax.fill_between(xs, mu-2*std, mu+2*std, alpha=0.2, color="steelblue", label="±2σ(x)")
ax.scatter(X_train.squeeze(), y_train, color="black", s=80, zorder=5, label="Observations")
ax.axhline(f_best, color="tomato", linestyle=":", linewidth=1.5, label=f"f_best = {f_best:.2f}")

low_mu_idx   = np.argmin(mu)
high_std_idx = np.argmax(std)
ax.annotate("Exploitation zone\n(low μ)",
            xy=(xs[low_mu_idx], mu[low_mu_idx]),
            xytext=(xs[low_mu_idx]+0.12, mu[low_mu_idx]+4),
            arrowprops=dict(arrowstyle="->", color="green"),
            color="green", fontsize=9)
ax.annotate("Exploration zone\n(high σ)",
            xy=(xs[high_std_idx], mu[high_std_idx]),
            xytext=(xs[high_std_idx]-0.28, mu[high_std_idx]-5),
            arrowprops=dict(arrowstyle="->", color="purple"),
            color="purple", fontsize=9)

ax.set_title("GP posterior — where should we evaluate next?")
ax.set_xlabel("x"); ax.set_ylabel("f(x)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 2 — Probability of Improvement (PI)

$$\text{PI}(x) = P(f(x) < f^* - \xi) = \Phi\!\left(\frac{f^* - \xi - \mu(x)}{\sigma(x)}\right)$$

- $\Phi$ is the standard normal CDF
- $f^*$ is the best value observed so far
- $\xi = 0$: greedy — next point is wherever the probability of being lower than $f^*$ is highest
- $\xi > 0$: requires improvement by at least $\xi$, pushing the next point into more uncertain regions

In [ ]:
xi_vals = [0.0, 0.05, 0.1]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, xi in enumerate(xi_vals):
    pi_vals = acq_pi(mu, std, f_best, xi=xi)
    draw_panel(axes[0, col], axes[1, col], mu, std, pi_vals,
               title=f"PI   ξ = {xi}", acq_label="PI(x)")

plt.suptitle("Probability of Improvement (PI)\n"
             "ξ=0 is greedy (over-exploits)   ξ>0 forces exploration", fontsize=12)
plt.tight_layout()
plt.show()

**Try it:** Set `xi=0` and move `f_best` to be very close to the true minimum (e.g., `f_best = -7.0`). Watch PI collapse — the algorithm gets stuck exploiting a tiny region near the best point.

## Part 3 — Expected Improvement (EI)

$$\text{EI}(x) = \mathbb{E}[\max(f^* - \xi - f(x),\, 0)]$$

Closed form (when $f(x) \sim \mathcal{N}(\mu, \sigma^2)$):

$$\text{EI}(x) = (f^* - \xi - \mu)\,\Phi(Z) + \sigma\,\phi(Z), \quad Z = \frac{f^* - \xi - \mu}{\sigma}$$

EI measures the **expected size** of improvement, not just the probability. So it automatically down-weights points that might improve but only by a tiny amount. This gives a better exploration–exploitation trade-off than PI.

In [ ]:
xi_vals = [0.0, 0.01, 0.1]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, xi in enumerate(xi_vals):
    ei_vals = acq_ei(mu, std, f_best, xi=xi)
    draw_panel(axes[0, col], axes[1, col], mu, std, ei_vals,
               title=f"EI   ξ = {xi}", acq_label="EI(x)")

plt.suptitle("Expected Improvement (EI)\n"
             "Weights improvement by its size AND probability — best default choice", fontsize=12)
plt.tight_layout()
plt.show()

**Try it:** Change the kernel `length_scale` to 0.5 (smoother GP). Does EI still suggest a sensible next point? Does the uncertainty band change shape and shift the peak of EI?

## Part 4 — Lower Confidence Bound (LCB / UCB)

$$\text{LCB}(x) = \mu(x) - \kappa\,\sigma(x)$$

We **minimize** LCB — the point that is simultaneously low in mean and high in uncertainty.

- $\kappa = 0$: pure exploitation (just minimise the GP mean)
- $\kappa \to \infty$: pure exploration (just go where the GP is most uncertain)
- $\kappa = 2.0$: standard default; roughly "optimistic under uncertainty"

UCB/LCB has strong theoretical regret bounds — this is why it is popular in the ML literature.

In [ ]:
kappa_vals = [0.5, 2.0, 4.0]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, kappa in enumerate(kappa_vals):
    lcb_vals = acq_lcb(mu, std, kappa=kappa)
    draw_panel(axes[0, col], axes[1, col], mu, std, lcb_vals,
               title=f"LCB   κ = {kappa}", acq_label="−LCB(x) = κσ−μ")

plt.suptitle("Lower Confidence Bound (LCB)\n"
             "LCB = μ − κσ   |   κ=0.5 exploits, κ=4.0 explores", fontsize=12)
plt.tight_layout()
plt.show()

**Try it:** Set `kappa=10.0`. The LCB next point will be far from all training data. Is this useful? This shows why extremely large κ wastes budget on random exploration.

## Part 5 — All three compared

With the same GP posterior, how different are the suggestions from PI, EI, and LCB?

In [ ]:
pi_vals  = acq_pi(mu, std, f_best, xi=0.01)
ei_vals  = acq_ei(mu, std, f_best, xi=0.01)
lcb_vals = acq_lcb(mu, std, kappa=2.0)

x_next_pi  = x_fine[np.argmax(pi_vals)].item()
x_next_ei  = x_fine[np.argmax(ei_vals)].item()
x_next_lcb = x_fine[np.argmax(lcb_vals)].item()

def normalise(v):
    v = v - v.min()
    return v / (v.max() + 1e-9)

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

axes[0].plot(xs, y_true, color="gray", linestyle="--", linewidth=1.5, alpha=0.5, label="True")
axes[0].plot(xs, mu, color="steelblue", linewidth=2, label="GP mean")
axes[0].fill_between(xs, mu-2*std, mu+2*std, alpha=0.15, color="steelblue")
axes[0].scatter(X_train.squeeze(), y_train, color="black", s=60, zorder=5, label="Data")
axes[0].axvline(x_next_pi,  color="tomato",  linewidth=2, linestyle="--", label=f"PI  next={x_next_pi:.3f}")
axes[0].axvline(x_next_ei,  color="green",   linewidth=2, linestyle="--", label=f"EI  next={x_next_ei:.3f}")
axes[0].axvline(x_next_lcb, color="purple",  linewidth=2, linestyle="--", label=f"LCB next={x_next_lcb:.3f}")
axes[0].set_title("GP posterior — suggested next point per acquisition function")
axes[0].set_ylabel("f(x)"); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].plot(xs, normalise(pi_vals),  color="tomato",  linewidth=2, label="PI  (ξ=0.01)")
axes[1].plot(xs, normalise(ei_vals),  color="green",   linewidth=2, label="EI  (ξ=0.01)")
axes[1].plot(xs, normalise(lcb_vals), color="purple",  linewidth=2, label="−LCB (κ=2.0)")
axes[1].axvline(x_next_pi,  color="tomato",  linewidth=1.5, linestyle="--")
axes[1].axvline(x_next_ei,  color="green",   linewidth=1.5, linestyle="--")
axes[1].axvline(x_next_lcb, color="purple",  linewidth=1.5, linestyle="--")
axes[1].set_title("Normalised acquisition functions — overlaid")
axes[1].set_xlabel("x"); axes[1].set_ylabel("Acquisition (normalised)")
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.suptitle("PI vs EI vs LCB — same GP, different next-point suggestions", fontsize=11)
plt.tight_layout()
plt.show()

print(f"PI  next: {x_next_pi:.3f}")
print(f"EI  next: {x_next_ei:.3f}")
print(f"LCB next: {x_next_lcb:.3f}")

## Part 6 — One complete BO step

Now we execute the full loop once:
1. Pick $x_{\text{next}} = \arg\max \text{EI}(x)$
2. Evaluate the real expensive function: $y_{\text{next}} = f(x_{\text{next}})$
3. Add $(x_{\text{next}}, y_{\text{next}})$ to the training set
4. Refit the GP

Lesson 8 wraps this into a full loop over many iterations.

In [ ]:
ei_for_step = acq_ei(mu, std, f_best, xi=0.01)
x_next      = x_fine[np.argmax(ei_for_step)]
y_next      = forrester(x_next)

X_new    = np.vstack([X_train, x_next])
y_new    = np.append(y_train, y_next)
f_best_new = y_new.min()

mu_new, std_new = gp_predict(X_new, y_new, x_fine, kern, noise=0.01)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Before
axes[0].plot(xs, y_true, "gray", linestyle="--", linewidth=1.5, alpha=0.5, label="True")
axes[0].plot(xs, mu, color="steelblue", linewidth=2, label="GP mean")
axes[0].fill_between(xs, mu-2*std, mu+2*std, alpha=0.2, color="steelblue", label="±2σ")
axes[0].scatter(X_train.squeeze(), y_train, color="black", s=60, zorder=5, label=f"Data (n={len(X_train)})")
axes[0].set_title("Step 1: GP posterior before")
axes[0].set_xlabel("x"); axes[0].set_ylabel("f(x)")
axes[0].legend(fontsize=7); axes[0].grid(True, alpha=0.3)

# Acquisition
axes[1].plot(xs, ei_for_step, color="darkorange", linewidth=2, label="EI(x)")
axes[1].fill_between(xs, 0, ei_for_step, alpha=0.25, color="darkorange")
axes[1].axvline(x_next.item(), color="red", linewidth=2.5, linestyle=":")
axes[1].set_title(f"Step 2: EI → x_next = {x_next.item():.3f}\nf(x_next) = {y_next.item():.3f}")
axes[1].set_xlabel("x"); axes[1].set_ylabel("EI(x)")
axes[1].legend(fontsize=7); axes[1].grid(True, alpha=0.3)

# After
axes[2].plot(xs, y_true, "gray", linestyle="--", linewidth=1.5, alpha=0.5, label="True")
axes[2].plot(xs, mu_new, color="steelblue", linewidth=2, label="GP mean (updated)")
axes[2].fill_between(xs, mu_new-2*std_new, mu_new+2*std_new, alpha=0.2, color="steelblue", label="±2σ")
axes[2].scatter(X_train.squeeze(), y_train, color="black", s=60, zorder=5, label="Old data")
axes[2].scatter([x_next.item()], [y_next.item()], color="red", s=150,
                zorder=6, marker="*", label="New point")
axes[2].set_title(f"Step 3: GP updated (n={len(X_new)})\nf_best: {f_best:.3f} → {f_best_new:.3f}")
axes[2].set_xlabel("x"); axes[2].set_ylabel("f(x)")
axes[2].legend(fontsize=7); axes[2].grid(True, alpha=0.3)

plt.suptitle("One complete Bayesian Optimisation step\n"
             "GP → EI → evaluate → update — Lesson 8 runs this in a loop", fontsize=11)
plt.tight_layout()
plt.show()

print(f"x_next   = {x_next.item():.4f}")
print(f"f(x_next) = {y_next.item():.4f}")
print(f"f_best before: {f_best:.4f}   after: {f_best_new:.4f}")

## Summary

| Acquisition | Formula | Default param | Strength |
|---|---|---|---|
| **PI** | $\Phi(Z)$ | ξ = 0.01 | Simple, easy to interpret |
| **EI** | $(f^*-\xi-\mu)\Phi(Z)+\sigma\phi(Z)$ | ξ = 0.01 | Best all-round default |
| **LCB** | $\mu - \kappa\sigma$ | κ = 2.0 | Theoretically grounded |

All three give the same next point roughly when the GP is well-calibrated. EI is the standard default for GP-BO.

**Next lesson:** Wrap this into a full Bayesian Optimisation loop — run 20 iterations on Forrester, track the best value found, and plot the convergence curve.